In [2]:
# Verifying data path

import os
import glob
import pandas as pd

paths = glob.glob("data/raw/**/*", recursive=True)

files = []
for item in paths:
    if os.path.isfile(item):
        files.append(item)

length = len(files)

print(files)
print(f"\n total files: {length}")

['data/raw\\delay\\TTC Subway Delay Data since 2025.csv', 'data/raw\\delay\\ttc-subway-delay-data-2018.xlsx', 'data/raw\\delay\\ttc-subway-delay-data-2019.xlsx', 'data/raw\\delay\\ttc-subway-delay-data-2020.xlsx', 'data/raw\\delay\\ttc-subway-delay-data-2021.xlsx', 'data/raw\\delay\\ttc-subway-delay-data-2022.xlsx', 'data/raw\\delay\\ttc-subway-delay-data-2023.xlsx', 'data/raw\\delay\\ttc-subway-delay-data-2024.xlsx', 'data/raw\\delay\\ttc-subway-delay-jan-2014-april-2017.xlsx', 'data/raw\\delay\\ttc-subway-delay-may-december-2017.xlsx', 'data/raw\\ridership\\1985-2019 Analysis of ridership.xlsx', 'data/raw\\ridership\\ttc-subway-station-usage-2012-2013.xlsx', 'data/raw\\ridership\\ttc-subway-station-usage-2014.xlsx', 'data/raw\\ridership\\ttc-subway-station-usage-2015.xlsx', 'data/raw\\ridership\\ttc-subway-station-usage-2016.xlsx', 'data/raw\\ridership\\ttc-subway-station-usage-2017.xlsx', 'data/raw\\schedules\\agency.txt', 'data/raw\\schedules\\calendar.txt', 'data/raw\\schedules\\c

In [3]:
# Verifying routes 

import duckdb

con = duckdb.connect("ttc.duckdb")


con.sql("CREATE OR REPLACE TABLE routes AS SELECT * " \
"FROM read_csv_auto('data/raw/schedules/routes.txt')")


con.sql("SELECT COUNT(route_id) FROM routes GROUP BY route_type").df()



,count(route_id)
0,20
1,3
2,210


In [4]:
con.sql("SELECT route_id, route_short_name, route_long_name " \
"  FROM routes " \
"WHERE route_type = 1").df()

,route_id,route_short_name,route_long_name
0,1,1,Line 1 (Yonge-University)
1,2,2,Line 2 (Bloor - Danforth)
2,4,4,Line 4 (Sheppard)


In [5]:
con.sql("CREATE OR REPLACE TABLE stops AS SELECT * " \
"FROM read_csv_auto('data/raw/schedules/stops.txt')")

con.sql("CREATE OR REPLACE TABLE stop_times AS SELECT * " \
"FROM read_csv_auto('data/raw/schedules/stop_times.txt')")

con.sql("CREATE OR REPLACE TABLE trips AS SELECT * " \
"FROM read_csv_auto('data/raw/schedules/trips.txt')")

In [6]:
# filter in only stop time data related to subway lines

con.sql("""
    CREATE OR REPLACE TABLE subway_stop_times AS
    SELECT stop_times.* FROM trips
    JOIN stop_times 
        ON trips.trip_id = stop_times.trip_id
    WHERE trips.route_id IN [1,2,4]
""")

con.sql("""
    SELECT * FROM subway_stop_times LIMIT 5
""").df()

,trip_id,arrival_time,departure_time,stop_id,stop_sequence,stop_headsign,pickup_type,drop_off_type,shape_dist_traveled
0,50659450,8:29:48,8:29:48,14945,1,None,0,0,NaN
1,50659450,8:32:14,8:32:14,15664,2,None,0,0,1.4442
2,50659450,8:34:52,8:34:52,15659,3,None,0,0,3.3901
3,50659450,8:37:04,8:37:04,15666,4,None,0,0,4.7396
4,50659450,8:38:56,8:38:56,15656,5,None,0,0,5.5524


In [7]:
con.sql("""
    SELECT * FROM stops LIMIT 5
""").df()

,stop_id,stop_code,stop_name,stop_desc,stop_lat,stop_lon,zone_id,stop_url,location_type,parent_station,stop_timezone,wheelchair_boarding
0,662,662,Danforth Rd at Kennedy Rd,None,43.714379,-79.260939,None,None,None,None,None,1
1,929,929,Davenport Rd at Bedford Rd,None,43.674448,-79.399659,None,None,None,None,None,1
2,940,940,Davenport Rd at Dupont St,None,43.675511,-79.401938,None,None,None,None,None,2
3,1871,1871,Davisville Ave at Cleveland St,None,43.702088,-79.378112,None,None,None,None,None,1
4,11700,11700,Disco Rd at Attwell Dr,None,43.701362,-79.594843,None,None,None,None,None,1


In [8]:
## fixing name duplicate i.e Bloor-Yonge

pd.set_option('display.max_rows', 70)

mapping = pd.DataFrame([
    ("Bloor Station", "Bloor-Yonge Station"),
    ("Yonge Station", "Bloor-Yonge Station")
], columns = ["raw_name", "canonical_name"])

con.sql("""
    CREATE OR REPLACE TABLE station_trips AS
        SELECT 
            COALESCE(mapping.canonical_name, SPLIT_PART(stop_name, ' -', 1)) AS stations, 
            COUNT(*) AS scheduled_trips
        FROM stops
        JOIN subway_stop_times
            ON subway_stop_times.stop_id = stops.stop_id
        LEFT JOIN mapping ON SPLIT_PART(stop_name, ' -', 1) = mapping.raw_name
        GROUP BY 1
        ORDER BY stations ASC
""")

con.sql("""
    CREATE OR REPLACE TABLE station_trips_clean AS
        WITH 
        s1 AS (
            SELECT * REPLACE (SPLIT_PART(stations, ' Station' , 1) AS stations) FROM station_trips),
        s2 AS (
            SELECT * REPLACE (UPPER(stations) AS stations) FROM s1),
        s3 AS (
            SELECT * RENAME (stations AS station) FROM s2)
    SELECT * FROM s3
""")

con.sql("""SELECT * FROM station_trips_clean""").df()


,station,scheduled_trips
0,BATHURST,2137
1,BAY,2136
2,BAYVIEW,1755
3,BESSARION,1758
4,BLOOR-YONGE,4332
5,BROADVIEW,2133
6,CASTLE FRANK,2133
7,CEDARVALE,2181
8,CHESTER,2133
9,CHRISTIE,2139


In [9]:
## Verifying delay datas

paths = glob.glob("data/raw/delay/**")

for f in paths:
    if os.path.splitext(f)[1].lower() == '.xlsx':
        df = pd.read_excel(f, nrows=0)
    else:
        df = pd.read_csv(f, nrows=0)
    print(f, df.columns.tolist())

path = "data/raw/delay/ttc-subway-delay-jan-2014-april-2017.xlsx"
sheets = pd.read_excel(path, sheet_name=None)
print(sheets.keys())

print(sheets['Incidents']['Date'].min())
print(sheets['Incidents']['Date'].max())

data/raw/delay\TTC Subway Delay Data since 2025.csv ['_id', 'Date', 'Time', 'Day', 'Station', 'Code', 'Min Delay', 'Min Gap', 'Bound', 'Line', 'Vehicle']
data/raw/delay\ttc-subway-delay-data-2018.xlsx ['Date', 'Time', 'Day', 'Station', 'Code', 'Min Delay', 'Min Gap', 'Bound', 'Line', 'Vehicle']
data/raw/delay\ttc-subway-delay-data-2019.xlsx ['Date', 'Time', 'Day', 'Station', 'Code', 'Min Delay', 'Min Gap', 'Bound', 'Line', 'Vehicle']
data/raw/delay\ttc-subway-delay-data-2020.xlsx ['Date', 'Time', 'Day', 'Station', 'Code', 'Min Delay', 'Min Gap', 'Bound', 'Line', 'Vehicle']
data/raw/delay\ttc-subway-delay-data-2021.xlsx ['Date', 'Time', 'Day', 'Station', 'Code', 'Min Delay', 'Min Gap', 'Bound', 'Line', 'Vehicle']
data/raw/delay\ttc-subway-delay-data-2022.xlsx ['Date', 'Time', 'Day', 'Station', 'Code', 'Min Delay', 'Min Gap', 'Bound', 'Line', 'Vehicle']
data/raw/delay\ttc-subway-delay-data-2023.xlsx ['Date', 'Time', 'Day', 'Station', 'Code', 'Min Delay', 'Min Gap', 'Bound', 'Line', 'Vehi

In [10]:
paths = glob.glob("data/raw/delay/**")

dfs = []

for f in paths:
    if os.path.splitext(f)[1].lower() == '.xlsx':
        df = pd.read_excel(f)
    else:
        df = pd.read_csv(f)
    dfs.append(df)

all_delays = pd.concat(dfs, ignore_index=True)

## all_delays.shape

(all_delays.dtypes.to_string())

## print(all_delays['Date'].max())
## print(all_delays['Date'].min())

'_id          float64\nDate          object\nTime             str\nDay              str\nStation          str\nCode             str\nMin Delay      int64\nMin Gap        int64\nBound            str\nLine             str\nVehicle        int64'

In [11]:
pd.set_option('display.max_rows', 100)

con.sql("""
CREATE OR REPLACE TABLE raw_delays AS
SELECT * FROM all_delays 
""")

con.sql("""
WITH ranked AS (
    SELECT
        Station, 
        COUNT(Station) AS count,
        count / SUM(COUNT(*)) OVER () AS ratio,
        SUM(COUNT(*)) OVER (ORDER BY COUNT(*) DESC) as r_total,
        r_total / SUM(COUNT(*)) OVER () AS r_ratio,
        ROW_NUMBER() OVER (ORDER BY COUNT(*) DESC) AS r_num
    FROM raw_delays
    GROUP BY Station
    ORDER BY Count DESC
)
SELECT * FROM ranked WHERE r_num IN (70,150)
""").df()



,Station,count,ratio,r_total,r_ratio,r_num
0,RUNNYMEDE STATION,1046,0.005596,168234.0,0.900051,70
1,VMC STATION PLATFORM 2,19,0.000102,183545.0,0.981965,150


In [12]:
con.sql("""
SELECT * FROM all_delays
""").df()

,_id,Date,Time,Day,Station,Code,Min Delay,Min Gap,Bound,Line,Vehicle
0,1.0,2025-01-01,02:10,Wednesday,BATHURST STATION,MUSAN,5,9,E,BD,5227
1,2.0,2025-01-01,02:30,Wednesday,DUNDAS STATION,MUIRS,0,0,NaN,YU,0
2,3.0,2025-01-01,02:32,Wednesday,BROADVIEW STATION,PUMST,0,0,E,BD,0
3,4.0,2025-01-01,02:58,Wednesday,KEELE STATION,EUSC,0,0,W,BD,5293
4,5.0,2025-01-01,02:58,Wednesday,COXWELL STATION,SUAE,0,0,NaN,BD,0
...,...,...,...,...,...,...,...,...,...,...,...
186911,NaN,2017-05-31 00:00:00,23:14,Wednesday,WILSON STATION (EXITIN,TUMVS,0,0,N,YU,5736
186912,NaN,2017-05-31 00:00:00,23:23,Wednesday,ST CLAIR WEST,SUDP,0,0,N,YU,5821
186913,NaN,2017-05-31 00:00:00,23:30,Wednesday,YONGE UNIVERSITY SUBWA,MUGD,0,0,NaN,YU,0
186914,NaN,2017-05-31 00:00:00,23:41,Wednesday,KENNEDY SRT STATION,MRTO,4,11,S,SRT,3021


In [ ]:
pd.set_option('display.max_rows', 25)

con.sql("""
CREATE OR REPLACE TABLE clean_delays AS 
    WITH 
    s2 AS (
        SELECT * REPLACE (SPLIT_PART(Station, ' STATION', 1) AS Station) FROM raw_delays),
    s3 AS (
        SELECT * REPLACE (REGEXP_REPLACE(Station, ' (BD|YU|YUS|SRT)$', '') AS Station) FROM s2),
    s4 AS (
        SELECT * REPLACE (REGEXP_REPLACE(Station, 'SHEPPARDSTATION$', 'SHEPPARD-YONGE') AS Station) FROM s3),
    s5 AS (
        SELECT * REPLACE (REGEXP_REPLACE(Station, 'BLOOR YONGE', 'BLOOR-YONGE') AS Station) FROM s4),
    s6 AS (
        SELECT * RENAME (Station AS station) FROM s5),
    s7 AS (
        SELECT * REPLACE (REPLACE(station, '.', '') AS station) FROM s6),
    s8 AS (
        SELECT * REPLACE (REPLACE(station, 'EGLINTON WEST', 'CEDARVALE') AS station) FROM s7),
    s9 AS (
        SELECT * REPLACE (REGEXP_REPLACE(station, '^DUNDAS$', 'TMU') AS station) FROM s8),
    s10 AS (
        SELECT * REPLACE (REGEXP_REPLACE(station, '^DUNDAS$', 'TMU') AS station) FROM s9),
    s11 AS (
        SELECT * REPLACE (REGEXP_REPLACE(station, '^BLOOR$', 'BLOOR-YONGE') AS station) FROM s10),
    s12 AS (
        SELECT * REPLACE (REGEXP_REPLACE(station, '^YONGE$', 'BLOOR-YONGE') AS station) FROM s11),
    s13 AS (
        SELECT * REPLACE (REGEXP_REPLACE(station, '^SHEPPARD$', 'SHEPPARD-YONGE') AS station) FROM s12),
    s14 AS (
        SELECT * REPLACE (REGEXP_REPLACE(station, '^VAUGHAN MC$', 'VAUGHAN METROPOLITAN CENTRE') AS station) FROM s13),
    s15 AS (
        SELECT * REPLACE (REPLACE(station, 'CTR', 'CENTRE') AS station) FROM s14),
    s16 AS (
        SELECT * REPLACE (REPLACE(station, 'VMC', 'VAUGHAN METROPOLITAN CENTRE') AS station) FROM s15),
    s17 AS (
        SELECT * REPLACE (REPLACE(station, 'YONGE SHP', 'SHEPPARD-YONGE') AS station) FROM s16),
    s18 AS (
        SELECT * REPLACE (REGEXP_REPLACE(station, ' STATIO$', '') AS station) FROM s17),
    s19 AS (
        SELECT * EXCLUDE(Date), SPLIT_PART(Date, ' ', 1) AS Date_normalized FROM s18),
    s20 AS (
        SELECT * EXCLUDE(_id, Time, Date_normalized), STRPTIME(Date_normalized || ' ' || Time, '%Y-%m-%d %H:%M') AS Timestamp FROM s19),
    s21 AS (
        SELECT * EXCLUDE(t1.station, scheduled_trips) FROM s20 LEFT JOIN station_trips_clean t1 ON s20.station = t1.station 
            WHERE t1.station IS NOT NULL)
SELECT * FROM s21
""")

result = con.sql("""
DESCRIBE clean_delays
""").df()

result



,column_name,column_type,null,key,default,extra
0,Day,VARCHAR,YES,None,None,None
1,station,VARCHAR,YES,None,None,None
2,Code,VARCHAR,YES,None,None,None
3,Min Delay,BIGINT,YES,None,None,None
4,Min Gap,BIGINT,YES,None,None,None
5,Bound,VARCHAR,YES,None,None,None
6,Line,VARCHAR,YES,None,None,None
7,Vehicle,BIGINT,YES,None,None,None
8,Timestamp,TIMESTAMP,YES,None,None,None


In [14]:
pd.set_option('display.max_rows', 100)

con.sql("""
WITH ranked AS (
    SELECT
        Station, 
        COUNT(Station) AS count,
        count / SUM(COUNT(*)) OVER () AS ratio,
        SUM(COUNT(*)) OVER (ORDER BY COUNT(*) DESC) as r_total,
        r_total / SUM(COUNT(*)) OVER () AS r_ratio,
        ROW_NUMBER() OVER (ORDER BY COUNT(*) DESC) AS r_num
    FROM clean_delays
    GROUP BY Station
    ORDER BY Count DESC
)
SELECT * FROM ranked WHERE r_num IN (70,150)
""").df()

,station,count,ratio,r_total,r_ratio,r_num
0,HIGHWAY 407,743,0.003975,177838.0,0.951433,70
1,YONGE- UNIVERSITY AND,9,0.000048,185425.0,0.992023,150


In [30]:
con.sql("""
SELECT COUNT(*) FROM clean_delays
WHERE Timestamp IS NULL
""").df()

,count_star()
0,0


In [16]:
pd.set_option('display.min_rows', 5)

con.sql("""
SELECT *
FROM clean_delays
""").df()

,_id,Time,Day,station,Code,Min Delay,Min Gap,Bound,Line,Vehicle,Date_normalized
0,1.0,02:10,Wednesday,BATHURST,MUSAN,5,9,E,BD,5227,2025-01-01
1,2.0,02:30,Wednesday,TMU,MUIRS,0,0,NaN,YU,0,2025-01-01
...,...,...,...,...,...,...,...,...,...,...,...
186914,NaN,23:41,Wednesday,KENNEDY,MRTO,4,11,S,SRT,3021,2017-05-31
186915,NaN,23:59,Wednesday,GLENCAIRN,MUSC,0,0,S,YU,5726,2017-05-31


In [45]:
pd.set_option('display.min_rows', 10)
pd.set_option('display.max_rows', 10)

con.sql("""
SELECT t1.station, line, COUNT(*) AS count
FROM clean_delays AS t1 
LEFT JOIN station_trips_clean t2 
    ON t1.station = t2.station
WHERE t2.station IS NULL AND t1.station LIKE '% TO %' AND LINE NOT LIKE 'SRT'
GROUP BY t1.station, line 
ORDER BY count DESC
""").df()




,station,Line,count


In [46]:
con.sql("""
SELECT t2.station, COUNT(*) AS count
FROM clean_delays AS t1 
RIGHT JOIN station_trips_clean t2 
    ON t1.station = t2.station
WHERE t1.station IS NOT NULL
GROUP BY t2.station
ORDER BY station ASC
""").df()


,station,count
0,BATHURST,2047
1,BAY,1468
2,BAYVIEW,856
3,BESSARION,505
4,BLOOR-YONGE,9543
...,...,...
65,WILSON,3759
66,WOODBINE,1989
67,YORK MILLS,2316
68,YORK UNIVERSITY,459


In [19]:
con.sql("""
SELECT COUNT(*) AS count
FROM clean_delays AS t1 
LEFT JOIN station_trips_clean t2 
    ON t1.station = t2.station
WHERE t2.station IS NULL
""").df()

,count
0,15296


In [57]:
pd.set_option('display.min_rows', 30)
pd.set_option('display.max_rows', 30)

con.sql("""
SELECT COUNT(*) FROM clean_delays 
WHERE "Min Delay" > 0
""").df()

,count_star()
0,60041


In [58]:
con.sql("""
SELECT COUNT(*) FROM clean_delays 
WHERE "Min Gap" > 0
""").df()

,count_star()
0,57883


In [155]:
## raw unreliability number

pd.set_option('display.min_rows', 40)
pd.set_option('display.max_rows', 40)

con.sql("""
CREATE OR REPLACE TABLE station_unreliability AS 
    WITH 
    s1 AS (
        SELECT station, 
            COUNT("Min Delay") AS delays, 
            SUM("Min Delay") AS total_delay, 
            AVG("Min Delay") AS avg_delay,
            PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY "Min Delay") AS p50,
            PERCENTILE_CONT(0.95) WITHIN GROUP (ORDER BY "Min Delay") AS p95
        FROM clean_delays 
        WHERE "Min Delay" > 0
        GROUP BY station),
    s2 AS (
        SELECT 
            t1.station, 
            delays / scheduled_trips AS delay_rate, 
            delays, total_delay, avg_delay,
            p50,
            p95
        FROM s1 t1 
        JOIN station_trips_clean t2 
            ON t1.station = t2.station)
    SELECT *, 
        RANK() OVER (ORDER BY delay_rate DESC) AS adjusted_rank, 
        RANK() OVER (ORDER BY delays DESC) AS raw_rank,
        raw_rank - adjusted_rank AS delta
    FROM s2
""")

In [114]:
con.sql("""
SELECT station, "Min Delay", Timestamp FROM clean_delays
WHERE "Min Delay" > 400
""").df()

,station,Min Delay,Timestamp
0,DUNDAS WEST,424,2014-09-30 07:55:00
1,KENNEDY,555,2015-02-07 15:30:00
2,COLLEGE,452,2015-03-24 05:59:00
3,JANE,575,2016-05-19 16:35:00
4,EGLINTON,807,2025-02-16 09:18:00
5,SHEPPARD WEST,900,2025-02-16 11:03:00
6,VICTORIA PARK,661,2026-01-25 15:21:00
7,GLENCAIRN,622,2026-01-25 16:12:00
8,KIPLING,441,2026-01-25 18:32:00
9,WOODBINE,827,2026-01-26 05:50:00


In [116]:
con.sql("""
SELECT MAX(Timestamp), MIN(Timestamp) FROM clean_delays
""").df()

,"max(""Timestamp"")","min(""Timestamp"")"
0,2026-06-30 23:55:00,2014-01-01 00:21:00


In [156]:
## with 2017 line 1 extension sensitivity consideration

pd.set_option('display.min_rows', 40)
pd.set_option('display.max_rows', 40)

con.sql("""
CREATE OR REPLACE TABLE station_unreliability_2018 AS 
    WITH 
    s1 AS (
        SELECT station, 
            COUNT("Min Delay") AS delays, 
            SUM("Min Delay") AS total_delay, 
            AVG("Min Delay") AS avg_delay,
            PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY "Min Delay") AS p50,
            PERCENTILE_CONT(0.95) WITHIN GROUP (ORDER BY "Min Delay") AS p95
        FROM clean_delays 
        WHERE "Min Delay" > 0 AND Timestamp > '2018-01-01'
        GROUP BY station),
    s2 AS (
        SELECT 
            t1.station, 
            delays / scheduled_trips AS delay_rate, 
            delays, total_delay, avg_delay,
            p50,
            p95
        FROM s1 t1 
        JOIN station_trips_clean t2 
            ON t1.station = t2.station)
    SELECT *, 
        RANK() OVER (ORDER BY delay_rate DESC) AS adjusted_rank, 
        RANK() OVER (ORDER BY delays DESC) AS raw_rank,
        raw_rank - adjusted_rank AS delta
    FROM s2
""")

In [84]:
con.sql("""
CREATE OR REPLACE TABLE calendar AS
    SELECT * FROM read_csv_auto('data/raw/schedules/calendar.txt')
""")

con.sql("""
SELECT * FROM calendar
""").df()

,service_id,monday,tuesday,wednesday,thursday,friday,saturday,sunday,start_date,end_date
0,1,1,1,1,1,1,0,0,20260726,20260905
1,2,0,0,0,0,0,1,0,20260726,20260905
2,3,0,0,0,0,0,0,1,20260726,20260905
3,4,0,0,0,0,0,0,0,20260726,20260905
4,501,0,0,0,0,0,0,0,20260726,20260905
5,6701,0,0,0,0,0,0,0,20260726,20260905
6,4401,0,0,0,0,0,0,0,20260726,20260905
7,4501,0,0,0,0,0,0,0,20260726,20260905
8,7001,0,0,0,0,0,0,0,20260726,20260905
9,6702,0,0,0,0,0,0,0,20260726,20260905


In [ ]:
con.sql("""
SELECT 
    t1.station, 
    t1.raw_rank - t2.raw_rank AS raw_diff,
    t1.adjusted_rank - t2.adjusted_rank AS adjusted_diff
FROM station_unreliability t1 
JOIN station_unreliability_2018 t2
    ON t1.station = t2.station
ORDER BY adjusted_diff DESC
""").df()

## the more positive the worst the delay

,station,raw_diff,adjusted_diff
0,HIGHWAY 407,20,21
1,FINCH WEST,14,16
2,DOWNSVIEW PARK,10,9
3,OSGOODE,10,8
4,NORTH YORK CENTRE,5,7
5,VAUGHAN METROPOLITAN CENTRE,6,6
6,LAWRENCE WEST,3,5
7,ST CLAIR,4,5
8,WELLESLEY,4,5
9,LAWRENCE,4,5


In [163]:
con.sql("""
SELECT station, adjusted_rank FROM station_unreliability
ORDER BY adjusted_rank ASC
LIMIT 10
""").df()


,station,adjusted_rank
0,KENNEDY,1
1,FINCH,2
2,KIPLING,3
3,EGLINTON,4
4,WILSON,5
5,BLOOR-YONGE,6
6,SHEPPARD WEST,7
7,DAVISVILLE,8
8,WARDEN,9
9,UNION,10


In [164]:
con.sql("""
SELECT station, adjusted_rank FROM station_unreliability_2018
ORDER BY adjusted_rank ASC
LIMIT 10
""").df()

,station,adjusted_rank
0,KENNEDY,1
1,FINCH,2
2,KIPLING,3
3,EGLINTON,4
4,WILSON,5
5,VAUGHAN METROPOLITAN CENTRE,6
6,BLOOR-YONGE,7
7,DAVISVILLE,8
8,UNION,9
9,ST GEORGE,10


In [165]:
con.sql("""
SELECT station, raw_rank FROM station_unreliability
ORDER BY raw_rank ASC
LIMIT 10
""").df()

,station,raw_rank
0,KENNEDY,1
1,BLOOR-YONGE,2
2,FINCH,3
3,KIPLING,4
4,EGLINTON,5
5,ST GEORGE,6
6,SHEPPARD-YONGE,7
7,WILSON,8
8,SHEPPARD WEST,9
9,DAVISVILLE,10


In [166]:
con.sql("""
SELECT station, raw_rank FROM station_unreliability_2018
ORDER BY raw_rank ASC
LIMIT 10
""").df()

,station,raw_rank
0,BLOOR-YONGE,1
1,KENNEDY,2
2,FINCH,3
3,KIPLING,4
4,ST GEORGE,5
5,EGLINTON,6
6,WILSON,7
7,VAUGHAN METROPOLITAN CENTRE,8
8,SHEPPARD-YONGE,9
9,SPADINA,10


In [168]:
con.sql("""
    SELECT * FROM station_trips_clean
""").df()

,station,scheduled_trips
0,BATHURST,2137
1,BAY,2136
2,BAYVIEW,1755
3,BESSARION,1758
4,BLOOR-YONGE,4332
5,BROADVIEW,2133
6,CASTLE FRANK,2133
7,CEDARVALE,2181
8,CHESTER,2133
9,CHRISTIE,2139


In [174]:
df = con.sql("""
    SELECT t1.station, delays, scheduled_trips
    FROM station_unreliability_2018 t1 
    JOIN station_trips_clean t2 ON t1.station = t2.station
""").df()

import numpy as np
rng = np.random.default_rng(1)

sims = rng.poisson(df['delays'].values,size=(5000, len(df)))

rates = sims / df['scheduled_trips'].values

df['ci_low'], df['ci_high'] = np.percentile(rates, [2.5, 97.5], axis=0)
df['rate'] = df['delays'] / df['scheduled_trips']

baseline = df['delays'].sum() / df['scheduled_trips'].sum()
df['worse_than_baseline'] = df['ci_low'] > baseline

df.sort_values('rate', ascending=False).head(15)

,station,delays,scheduled_trips,ci_low,ci_high,rate,worse_than_baseline
1,KENNEDY,1870,2158,0.827155,0.905931,0.866543,True
2,FINCH,1800,2224,0.772932,0.846223,0.809353,True
3,KIPLING,1618,2155,0.713689,0.788875,0.750812,True
5,EGLINTON,1575,2223,0.674314,0.743151,0.708502,True
6,WILSON,1245,2181,0.539649,0.602017,0.570839,True
7,VAUGHAN METROPOLITAN CENTRE,1127,2154,0.492572,0.553389,0.523213,True
0,BLOOR-YONGE,2172,4332,0.480609,0.522392,0.501385,True
10,DAVISVILLE,904,2190,0.385388,0.439269,0.412785,True
11,UNION,881,2195,0.375843,0.428702,0.401367,True
4,ST GEORGE,1578,4326,0.346278,0.383264,0.364771,True


In [ ]:
con.sql("""
    CREATE OR REPLACE TABLE station_ci AS
        SELECT * FROM df
""")

con.sql("""
    CREATE OR REPLACE TABLE station_data_final AS
        SELECT * EXCLUDE(t2.station, t2.delays, rate) FROM station_unreliability_2018 t1 
        JOIN station_ci t2 
            ON t1.station = t2.station 
""")

con.sql("""
    SELECT * FROM station_data_final
""").df()


con.sql("""
    COPY station_data_final TO 'output/station_final.csv' (HEADER, DELIMITER ',')
""")


CatalogException: Catalog Error: Table with name station_final does not exist!
Did you mean "station_ci"?